# Loan Default Prediction with Random Forest

## Business Problem
Loan default prediction is a binary classification problem in which the goal is to identify borrowers at elevated risk of default.

For a credit-risk use case, overall accuracy alone can be misleading because defaults are the minority class. A particularly important metric is **recall for the default class (`BAD = 1`)**, because a false negative represents a risky loan that the model fails to flag.

This notebook develops a leakage-safe Random Forest workflow and evaluates whether decision-threshold tuning can improve default detection.


## Dataset Overview

The dataset used in the original analysis contains **5,960 observations and 13 variables**.

- **Target:** `BAD`
- `BAD = 0`: non-default / good loan
- `BAD = 1`: default / bad loan
- Observed default rate: approximately **19.95%**

The dataset file is expected at `data/hmeq.csv`. The raw data is intentionally kept separate from the analysis code so the repository does not depend on a local computer path.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
DATA_PATH = Path("data") / "hmeq.csv"


## Load and Inspect the Data

Using a relative path makes the notebook portable. Anyone who clones the repository can reproduce the analysis by placing the dataset in the `data/` directory.


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()


In [ ]:
df.info()


In [ ]:
# Descriptive statistics for numeric variables
df.select_dtypes(include="number").describe().T


In [ ]:
# Descriptive statistics for categorical variables
df.select_dtypes(exclude="number").describe().T


## Data Quality Assessment

The original analysis found missing values in several predictors. `DEBTINC` had the highest missingness at approximately **21.26%**, followed by `DEROG` at approximately **11.88%** and `DELINQ` at approximately **9.73%**.

Rather than imputing the full dataset before the train/test split, the final model performs imputation **inside a scikit-learn pipeline fitted only on the training data**. This avoids allowing information from the test set to influence preprocessing.


In [ ]:
missing_summary = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename("missing_percent")
      .to_frame()
)

missing_summary


## Exploratory Data Analysis


In [ ]:
default_rate = df["BAD"].mean()

ax = sns.countplot(data=df, x="BAD")
ax.set_title(f"Loan Outcome Distribution — Default Rate: {default_rate:.1%}")
ax.set_xlabel("BAD (0 = Non-default, 1 = Default)")
ax.set_ylabel("Count")
plt.show()


In [ ]:
numeric_cols_eda = df.select_dtypes(include=np.number).columns

corr = df[numeric_cols_eda].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

corr["BAD"].drop("BAD").sort_values(ascending=False)


### EDA Findings from the Original Analysis

The strongest numeric relationships with default were:

| Variable | Correlation with `BAD` | Interpretation |
|---|---:|---|
| `DELINQ` | +0.354 | More delinquent credit lines were associated with higher default risk |
| `DEROG` | +0.276 | More major derogatory reports were associated with higher risk |
| `DEBTINC` | +0.200 | Higher debt-to-income burden was associated with higher risk |
| `NINQ` | +0.175 | More recent credit inquiries were associated with higher risk |
| `CLAGE` | -0.170 | Longer credit history was associated with lower risk |
| `LOAN` | -0.075 | Loan amount alone had relatively weak linear association with default |

These are **associations**, not causal effects.


In [ ]:
def plot_numeric_by_target(data, feature):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(
        data=data,
        x=feature,
        hue="BAD",
        kde=True,
        element="step",
        stat="density",
        common_norm=False,
        ax=axes[0],
    )
    axes[0].set_title(f"{feature} Distribution by Loan Outcome")

    sns.boxplot(data=data, x="BAD", y=feature, ax=axes[1])
    axes[1].set_title(f"{feature} by Loan Outcome")

    plt.tight_layout()
    plt.show()

for feature in ["DELINQ", "DEROG", "DEBTINC", "NINQ", "CLAGE"]:
    plot_numeric_by_target(df, feature)


## Leakage-Safe Train/Test Split

An earlier exploratory version of the analysis performed preprocessing before the split and produced unrealistically perfect test performance. That result was treated as a warning sign rather than accepted.

The final workflow starts again from the raw data, splits it first, and fits all learned preprocessing steps only on the training set.


In [ ]:
X = df.drop(columns="BAD")
y = df["BAD"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("Training default rate:", round(y_train.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))


## Preprocessing Pipeline

- Numeric predictors: median imputation
- Categorical predictors: most-frequent imputation followed by one-hot encoding
- Unknown categories at prediction time are ignored safely

Random Forest does not require feature scaling, so scaling is omitted from the final pipeline.


In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)


## Random Forest Model

Class weighting is used because defaults represent only about one-fifth of the observations.


In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)


## Baseline Evaluation — Default Threshold 0.50


In [ ]:
y_prob = rf_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Non-default", "Default"],
)
plt.title("Confusion Matrix — Threshold 0.50")
plt.show()


### Original Baseline Results

The original run produced:

- **Accuracy:** 90%
- **ROC-AUC:** 0.963
- **Default precision:** 0.87
- **Default recall:** 0.61
- **Default F1-score:** 0.72

The model discriminated well overall, but default recall of 61% meant that approximately **39% of actual defaults were not identified** at the standard 0.50 decision threshold.

For a risk-screening application, that motivates evaluating a lower threshold.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


## Decision-Threshold Tuning

Scikit-learn classifies a case as positive at a probability threshold of 0.50 by default. In credit-risk screening, a lower threshold can be appropriate when the cost of missing a potential default is greater than the cost of sending an additional borrower for review.

The original analysis evaluated a threshold of **0.35**.


In [ ]:
CUSTOM_THRESHOLD = 0.35
y_pred_custom = (y_prob >= CUSTOM_THRESHOLD).astype(int)

print(classification_report(y_test, y_pred_custom))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_custom,
    display_labels=["Non-default", "Default"],
)
plt.title(f"Confusion Matrix — Threshold {CUSTOM_THRESHOLD:.2f}")
plt.show()


### Threshold 0.35 Results from the Original Analysis

After lowering the threshold from 0.50 to 0.35:

- **Default recall increased from 61% to 79%**
- **Default precision was 79%**
- **Default F1-score was 0.79**
- **Overall accuracy was 92%**

The key result is the improvement in default recall. The model identified substantially more actual defaulters while retaining reasonably strong precision.

Threshold selection should ultimately be based on the organization's relative costs of false negatives and false positives rather than on a single performance metric.


## Key Takeaways

1. The dataset is imbalanced: approximately 20% of observations are defaults.
2. Behavioral credit variables such as delinquency, derogatory history, debt burden, and credit inquiries showed stronger relationships with default than loan amount alone.
3. The final modeling workflow prevents preprocessing leakage by splitting the raw data before fitting imputers and encoders.
4. The Random Forest achieved an original ROC-AUC of approximately **0.963**.
5. At the default 0.50 threshold, default recall was **61%**.
6. Lowering the threshold to 0.35 increased default recall to **79%**, illustrating the business importance of choosing a decision threshold based on risk objectives.


## Limitations and Next Steps

This project is a predictive modeling demonstration rather than a production credit-decision system.

Potential next steps include:

- Compare Random Forest with logistic regression and gradient-boosting models.
- Use cross-validation for more robust model comparison.
- Tune hyperparameters using training data only.
- Evaluate precision-recall curves and threshold trade-offs.
- Assess probability calibration.
- Examine feature importance and model explainability.
- Evaluate subgroup performance and fairness before any real-world lending application.
- Translate false-positive and false-negative errors into estimated business costs before selecting an operating threshold.


## Reproducibility

Expected repository structure:

```text
Loan_Default_Prediction/
├── README.md
├── Loan_Default_Prediction.ipynb
├── data/
│   └── hmeq.csv
├── requirements.txt
└── .gitignore
```

If the dataset should not be redistributed, exclude `data/hmeq.csv` from Git and document where authorized users can obtain it.
